<a href="https://colab.research.google.com/github/Qureshiii/PyTorch-Learning-Journey/blob/main/11_Question_Answering_System_QA_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/100_Unique_QA_Dataset.csv')
df

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100
...,...,...
85,Who directed the movie 'Titanic'?,JamesCameron
86,Which superhero is also known as the Dark Knight?,Batman
87,What is the capital of Brazil?,Brasilia
88,Which fruit is known as the king of fruits?,Mango


### Tokenization

In [ ]:
def tokenize(text):
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'",'')
  return text.split()

In [ ]:
tokenize("Who wrote 'To Kill a Mockingbird'?")

['who', 'wrote', 'to', 'kill', 'a', 'mockingbird']

### Vocabualary Formation

In [ ]:
vocab = {'<UNK>':0}

In [ ]:
def build_vocab(row):

  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merge_tokens = tokenized_question + tokenized_answer

  for token in merge_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

  print(merge_tokens)

In [ ]:
df.apply(build_vocab, axis=1)

['what', 'is', 'the', 'capital', 'of', 'france', 'paris']
['what', 'is', 'the', 'capital', 'of', 'germany', 'berlin']
['who', 'wrote', 'to', 'kill', 'a', 'mockingbird', 'harper-lee']
['what', 'is', 'the', 'largest', 'planet', 'in', 'our', 'solar', 'system', 'jupiter']
['what', 'is', 'the', 'boiling', 'point', 'of', 'water', 'in', 'celsius', '100']
['who', 'painted', 'the', 'mona', 'lisa', 'leonardo-da-vinci']
['what', 'is', 'the', 'square', 'root', 'of', '64', '8']
['what', 'is', 'the', 'chemical', 'symbol', 'for', 'gold', 'au']
['which', 'year', 'did', 'world', 'war', 'ii', 'end', '1945']
['what', 'is', 'the', 'longest', 'river', 'in', 'the', 'world', 'nile']
['what', 'is', 'the', 'capital', 'of', 'japan', 'tokyo']
['who', 'developed', 'the', 'theory', 'of', 'relativity', 'albert-einstein']
['what', 'is', 'the', 'freezing', 'point', 'of', 'water', 'in', 'fahrenheit', '32']
['which', 'planet', 'is', 'known', 'as', 'the', 'red', 'planet', 'mars']
['who', 'is', 'the', 'author', 'of', '19

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [ ]:
len(vocab)

324

### Convert Words to Numerical indices

In [ ]:
def text_to_indices(text, vocab):

  indexed_text = []

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [ ]:
text_to_indices('what is campusX',vocab)

[1, 2, 0]

### DataSet & DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
class QADataset(Dataset):

  def __init__(self,df,vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [ ]:
dataset = QADataset(df,vocab)
dataset[0]

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))

In [ ]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

for question, answer in dataloader:
  print(question, answer)

tensor([[  1,   2,   3,   4,   5, 135]]) tensor([[136]])
tensor([[ 42, 137,   2, 226,  12,   3, 227, 228]]) tensor([[155]])
tensor([[ 1,  2,  3, 24, 25,  5, 26, 19, 27]]) tensor([[28]])
tensor([[ 1,  2,  3, 37, 38, 39, 40]]) tensor([[41]])
tensor([[  1,   2,   3,   4,   5, 109]]) tensor([[317]])
tensor([[  1,   2,   3,  33,  34,   5, 245]]) tensor([[246]])
tensor([[1, 2, 3, 4, 5, 8]]) tensor([[9]])
tensor([[  1,   2,   3, 212,   5,  14, 213, 214]]) tensor([[215]])
tensor([[ 10,  29, 130, 131]]) tensor([[132]])
tensor([[ 42, 312,   2, 313,  62,  63,   3, 314, 315]]) tensor([[316]])
tensor([[  1,   2,   3, 163, 164, 165,  83,  84]]) tensor([[166]])
tensor([[ 1,  2,  3, 69,  5, 53]]) tensor([[260]])
tensor([[ 10, 140,   3, 141, 270,  93, 271,   5,   3, 272]]) tensor([[273]])
tensor([[ 78,  79, 261, 151,  14, 262, 153]]) tensor([[36]])
tensor([[  1,   2,   3, 221,   5, 222, 223, 224]]) tensor([[225]])
tensor([[ 10,  75, 111]]) tensor([[112]])
tensor([[  1,   2,   3, 234,   5, 235]]) tensor

In [ ]:
import torch.nn as nn

In [ ]:
class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):
    super(). __init__()

    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50,64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [ ]:
learning_rate = 0.001
epochs = 20

In [ ]:
model = SimpleRNN(len(vocab))

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

### Loop debugging code

In [ ]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [ ]:
# Training Loop

for epoch in range(epochs):

  total_loss = 0.0

  for question, answer in dataloader:
    optimizer.zero_grad()

    # forward pass

    output = model(question)

    # loss

    loss = criterion(output, answer[0])

    # Gradients
    loss.backward()

    # update
    optimizer.step()

    # calculate total loss

    total_loss = total_loss + loss.item()


  avg_loss = total_loss / len(dataloader)

  print(f'Epoch {epoch + 1}, loss: {avg_loss:.4f}')

Epoch 1, loss: 0.1120
Epoch 2, loss: 0.0986
Epoch 3, loss: 0.0873
Epoch 4, loss: 0.0776
Epoch 5, loss: 0.0696
Epoch 6, loss: 0.0626
Epoch 7, loss: 0.0568
Epoch 8, loss: 0.0517
Epoch 9, loss: 0.0472
Epoch 10, loss: 0.0432
Epoch 11, loss: 0.0396
Epoch 12, loss: 0.0364
Epoch 13, loss: 0.0336
Epoch 14, loss: 0.0311
Epoch 15, loss: 0.0288
Epoch 16, loss: 0.0268
Epoch 17, loss: 0.0249
Epoch 18, loss: 0.0232
Epoch 19, loss: 0.0217
Epoch 20, loss: 0.0203


### Doing Predictions

In [ ]:
def predict(model, question, threshold=0.5):

  # convert word to numbers

  numerical_question = text_to_indices(question, vocab)

  # convert to tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probabilites
  probabilities = torch.nn.functional.softmax(output, dim=1)

  # find index of max probabilities
  value, index = torch.max(probabilities, dim=1)


  if value < threshold:
    print("I don't Know")

  print(list(vocab.keys())[index])

In [ ]:
predict(model, "What is the boiling point of water in Celsius?")

100
